In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyreadstat
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, RepeatedKFold
from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.metrics import make_scorer, mean_absolute_error
from typing import Dict, Tuple, Union, List




In [ ]:
#code/functions for preprocessing PISA Dataset 

def LFM_data(student_path, school_path, country):  #load + filter + merge data
    print(f"Loading student and school data...")
    student_df, _ = pyreadstat.read_sav(student_path, apply_value_formats=False) 
    school_df, _ = pyreadstat.read_sav(school_path, apply_value_formats=False)

    print(f"Filtering student and school data for country...")
    student_df = student_df[student_df["CNT"] == country]
    school_df = school_df[school_df["CNT"] == country][["CNTSCHID","RATCMP1","RATCMP2","EDUSHORT"]]

    print(f"Merging student + school data...")
    return student_df.merge(school_df, on="CNTSCHID", how="left")

def process_data(df, predictors, dv):
    print(f"Keeping predictors + dv, declaring x and y, and encoding gender...")
    df = df[predictors + [dv, "CNTSCHID"]]

    X = df.drop(columns = [dv, "CNTSCHID"]) #drop DV + School ID and has only features
    y = df[dv].copy() #copy the dependent variable

    if "ST004D01T" in X.columns:
        X["ST004D01T"] = X["ST004D01T"].map({1: 0, 2: 1})

    print(f"Dropping columns with all missing values...")
    X = X.dropna(axis=1, how="all")  # drop columns where all values are NaN

    print(f"Imputing missing values and scaling features...")
    imputer = SimpleImputer(strategy="median")
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X.columns, index=X.index)

    final_df = X_scaled.copy()
    final_df[dv] = y.values
    return final_df

def save_data(df, output_path):
    print(f"Saving the final processed dataframe...") 
    df.to_pickle(output_path)
    print(f"Saved processed data → {output_path} (shape: {df.shape})")


def preprocess_pisa_data(student_path, school_path, country, predictors, dv, output_path):
    print(f"Preprocessing PISA data for {country}...")
    df = LFM_data(student_path, school_path, country)
    df = process_data(df, predictors, dv)
    save_data(df, output_path)
    print(f"Preprocessing complete.")


In [ ]:
#code/functions for data analysis and model training  

#function for curent best model 
def best_xgb():
    xgb = XGBRegressor(
    n_estimators = 300,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    reg_lambda = 4,
    reg_alpha = 0.6, 
    min_child_weight = 5,
    objective = "reg:squarederror",
    gamma = 0,
    random_state = 42,
    verbosity = 0
    )
    return xgb

#function to evaluate model performance
def evaluate_model(X, y, model):
    print(f"Evaluvating model performance...")
    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
    mae_scores = cross_val_score(model, X, y, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1)
    mae_scores = -mae_scores
    mean_mae = np.mean(mae_scores)

    print(f"Cross-validated MAE: {np.round(mean_mae, 3)}")


#function to find top 10 factors of BEINGBULLIED for any given country
def find_factors(path, model, country):
    print(f"Finding top 10 factors...")
    df = pd.read_pickle(path) 
    df = df.dropna(subset=["BEINGBULLIED"])
    X = df.drop(columns=["BEINGBULLIED"])
    y = df["BEINGBULLIED"]
    model.fit(X,y)
    importances = model.feature_importances_  # get feature importances
    feat_imp = sorted(zip(X.columns, importances), key=lambda pair: pair[1], reverse=True)
    print("Top 10 XGB predictors of BEINGBULLIED (",country,"):")
    for feat, imp in feat_imp[:10]:
        print(f"  {feat:<12s}: {imp:.4f}")

In [ ]:
STU_PATH = Path("../data/raw/Student Data.sav") #path to the student data file
SCH_PATH = Path("../data/raw/School Data.sav") #path to the school data file

PREDICTORS = [
    #Individual-level Predictors
    "ST004D01T", #Gender
    "AGE", #Age
    "GRADE", #Grade
    "BSMJ", #Expected Occupational Status
    "JOYREAD", #Joy of Reading
    "SCREADCOMP", #Reading Self-Concept: Competence
    "SCREADDIFF", #Reading Self-Concept: Difficulty
    "COMPETE", #Competitiveness
    "WORKMAST", #Work Mastery Orientation
    "GFOFAIL", # General Fear of Failure
    "EUDMO", #Sense of Meaning in Life (Eudaimonia)  
    "RESILIENCE", #Resilience
    "MASTGOAL", #Mastery Goal Orientation
    "ST185Q01HA", #Does life has meaning/purpose 
    "ST184Q01HA", #Growth Mindset
    "SWBP", #Well Being 
    "PV1MATH", #Math Performance
    "PV1READ", #Reading Performance
    "PV1SCIE", #Science Performance

    #Proximal-level Predictors
    "REPEAT", #Grade Repetition History
    "UNDREM", #Meta-cognition: Understanding & Remembering 
    "METASUM", #Meta-cognition: Summarizing
    "METASPAM", #Meta-cognition: Assessing Credibility

    #Microsystem-Level Factors (Family, Peers, & School CLimate)
    "EMOSUPS", #Parental Emotional Support
    "DURECEC", #Duration in Early Childhood Education and Care
    "BELONG", #School Belonging
    "PERCOMP", #Perceived School Competitiveness 
    "PERCOOP", #Perceived School Cooperation
    "ATTLNACT", #Attitudes Towards Learning Activities
    "DISCLIMA", #Disciplinary Climate (Language Lessons)
    "TEACHSUP", #Teacher Support (Language Lessons)
    "DIRINS", #Teacher-Directed Instruction 
    "PERFEED", #Perceived Feedback from Teachers
    "STIMREAD", #Teacher's Stimulation of Reading Engagement
    "ADAPTIVITY", #Adapation of Instruction
    "TEACHINT", #Perceived Teacher Interest 

    #Macrosystem/Exosystem-level Predictors
    "ESCS", #Family Socioeconomic Status(Index)
    "EDUSHORT", #Shortage of Educational Resources
    "RATCMP1", #Number of Computers per Student
    "RATCMP2", #Percentage of Computers Connected to the Internet
]

DV = "BEINGBULLIED"

countries = [
    ("JPN", "Japan"),
    ("KOR", "South Korea"),
    ("PHL", "Philippines"),
    ("THA", "Thailand"),
    ("GBR", "United Kingdom"),
    ("FIN", "Finland"),
    ("DEU", "Germany"),
    ("POL", "Poland"),
    ("ITA", "Italy"),
    ("USA", "United States"),
    ("CAN", "Canada"),
    ("MEX", "Mexico"),
    ("BRA", "Brazil"),
    ("ISR", "Israel"),
    ("JOR", "Jordan"),
    ("MAR", "Morocco"),
    ("SGP", "Singapore"),
    ("KAZ", "Kazakhstan"),
    ("AUS", "Australia"),
    ("NZL", "New Zealand")
]

for code, name in countries:
    output_path = Path(f"../data/processed/{code}_data.pkl")
    
    preprocess_pisa_data(STU_PATH, SCH_PATH, code, PREDICTORS, DV, output_path)
    find_factors(output_path, best_xgb(), name)



Preprocessing PISA data for JPN...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + dv, declaring x and y, and encoding gender...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\JPN_data.pkl (shape: (6109, 40))
Preprocessing complete.
Finding top 10 factors...
Top 10 XGB predictors of BEINGBULLIED ( Japan ):
  BELONG      : 0.0742
  ST004D01T   : 0.0500
  GFOFAIL     : 0.0428
  DISCLIMA    : 0.0369
  EMOSUPS     : 0.0322
  ESCS        : 0.0316
  PERCOOP     : 0.0307
  SCREADCOMP  : 0.0300
  METASPAM    : 0.0280
  PERCOMP     : 0.0276
Preprocessing PISA data for KOR...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + dv, declaring x and y, and encoding gender...
Dropping columns with all m

In [13]:
STU_PATH = Path("../data/raw/Student Data.sav") #path to the student data file
SCH_PATH = Path("../data/raw/School Data.sav") #path to the school data file

PREDICTORS = [
    #Individual-level Predictors
    "ST004D01T", #Gender
    "AGE", #Age
    "GRADE", #Grade
    "BSMJ", #Expected Occupational Status
    "JOYREAD", #Joy of Reading
    "SCREADCOMP", #Reading Self-Concept: Competence
    "SCREADDIFF", #Reading Self-Concept: Difficulty
    "COMPETE", #Competitiveness
    "WORKMAST", #Work Mastery Orientation
    "GFOFAIL", # General Fear of Failure
    "EUDMO", #Sense of Meaning in Life (Eudaimonia)  
    "RESILIENCE", #Resilience
    "MASTGOAL", #Mastery Goal Orientation
    "ST185Q01HA", #Does life has meaning/purpose 
    "ST184Q01HA", #Growth Mindset
    "SWBP", #Well Being 
    "PV1MATH", #Math Performance
    "PV1READ", #Reading Performance
    "PV1SCIE", #Science Performance

    #Proximal-level Predictors
    "REPEAT", #Grade Repetition History
    "UNDREM", #Meta-cognition: Understanding & Remembering 
    "METASUM", #Meta-cognition: Summarizing
    "METASPAM", #Meta-cognition: Assessing Credibility

    #Microsystem-Level Factors (Family, Peers, & School CLimate)
    "EMOSUPS", #Parental Emotional Support
    "DURECEC", #Duration in Early Childhood Education and Care
    "BELONG", #School Belonging
    "PERCOMP", #Perceived School Competitiveness 
    "PERCOOP", #Perceived School Cooperation
    "ATTLNACT", #Attitudes Towards Learning Activities
    "DISCLIMA", #Disciplinary Climate (Language Lessons)
    "TEACHSUP", #Teacher Support (Language Lessons)
    "DIRINS", #Teacher-Directed Instruction 
    "PERFEED", #Perceived Feedback from Teachers
    "STIMREAD", #Teacher's Stimulation of Reading Engagement
    "ADAPTIVITY", #Adapation of Instruction
    "TEACHINT", #Perceived Teacher Interest 

    #Macrosystem/Exosystem-level Predictors
    "ESCS", #Family Socioeconomic Status(Index)
    "EDUSHORT", #Shortage of Educational Resources
    "RATCMP1", #Number of Computers per Student
    "RATCMP2", #Percentage of Computers Connected to the Internet
]

DV = "BEINGBULLIED"

countries = [
    ("SGP", "Singapore"),
    ("KAZ", "Kazakhstan"),
    ("AUS", "Australia"),
    ("NZL", "New Zealand")
]

for code, name in countries:
    output_path = Path(f"../data/processed/{code}_data.pkl")
    
    preprocess_pisa_data(STU_PATH, SCH_PATH, code, PREDICTORS, DV, output_path)
    find_factors(output_path, best_xgb(), name)



Preprocessing PISA data for SGP...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + dv, declaring x and y, and encoding gender...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\SGP_data.pkl (shape: (6676, 38))
Preprocessing complete.
Finding top 10 factors...
Top 10 XGB predictors of BEINGBULLIED ( Singapore ):
  ST004D01T   : 0.1321
  BELONG      : 0.1188
  DISCLIMA    : 0.0509
  PERCOOP     : 0.0306
  GFOFAIL     : 0.0285
  PV1READ     : 0.0280
  PV1MATH     : 0.0272
  EMOSUPS     : 0.0263
  PERCOMP     : 0.0242
  COMPETE     : 0.0231
Preprocessing PISA data for KAZ...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + dv, declaring x and y, and encoding gender...
Dropping columns with a